# Notebook 11 – Imbalanced Data

This notebook uses `customer_transactions_raw.csv` to demonstrate class imbalance and treatment techniques, using **Platinum membership** as the minority class we want to predict (is this customer a Platinum member or not).

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('customer_transactions_raw.csv')
df['age_numeric'] = pd.to_numeric(df['age'], errors='coerce')
df['purchase_amount_clean'] = df['purchase_amount'].astype(str).str.replace('$', '', regex=False)
df['purchase_amount_clean'] = pd.to_numeric(df['purchase_amount_clean'], errors='coerce')
df['membership_clean'] = df['membership_type'].str.strip().str.lower()
df['membership_clean'].value_counts(dropna=False)

membership_clean
bronze      317
silver      313
gold        189
platinum    118
NaN          63
Name: count, dtype: int64

## 1. What is Class Imbalance?

Class imbalance occurs when the categories (classes) of a target variable are not represented in roughly equal proportions. One class (the **majority class**) has far more examples than another (the **minority class**), which makes learning to recognize the minority class harder.

In [2]:
df['is_platinum'] = (df['membership_clean'] == 'platinum').astype(int)
df['is_platinum'].value_counts()

is_platinum
0    882
1    118
Name: count, dtype: int64

In [3]:
df['is_platinum'].value_counts(normalize=True) * 100

is_platinum
0    88.2
1    11.8
Name: proportion, dtype: float64

## 2. Balanced vs Imbalanced Dataset

A **balanced** dataset has classes represented in roughly similar proportions (e.g. 50/50 or 60/40). An **imbalanced** dataset has a skewed ratio — here, Platinum members make up only about 12–13% of customers, an imbalance ratio of roughly 7:1 against them. Some real-world problems (fraud detection, rare disease diagnosis) can be far more extreme, with ratios of 100:1 or more — this dataset's `purchase_amount_clean > 500` flag, for instance, is true for only about 6 out of 1000 rows.

In [4]:
extreme_minority_example = (df['purchase_amount_clean'] > 500).astype(int)
extreme_minority_example.value_counts()

purchase_amount_clean
0    994
1      6
Name: count, dtype: int64

## 3. Why Class Imbalance is a Problem

- Most learning algorithms are trained to minimize overall error, so they naturally gravitate toward correctly predicting the majority class, since that's what most of the training examples look like.
- A model can achieve very high accuracy simply by **always predicting the majority class**, while being completely useless at identifying the minority class — which is often the class we actually care about most (fraud, churn, disease, high-value customers).
- The model effectively receives far less signal about the minority class's patterns, since there are fewer examples to learn from.

## 4. Why Accuracy Alone Can Be Misleading

With our ~87% majority-class rate, a model that predicts "not Platinum" for every single customer would already be about 87% accurate — despite never correctly identifying a single Platinum member.

In [5]:
majority_class_accuracy = 1 - df['is_platinum'].mean()
majority_class_accuracy

np.float64(0.882)

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
feature_cols = ['annual_income', 'quantity', 'rating']
X = df[feature_cols].fillna(df[feature_cols].median())
y = df['is_platinum']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, dummy_predictions))
print('Precision (minority class):', precision_score(y_test, dummy_predictions, zero_division=0))
print('Recall (minority class):', recall_score(y_test, dummy_predictions, zero_division=0))
print('F1 (minority class):', f1_score(y_test, dummy_predictions, zero_division=0))

Accuracy: 0.8833333333333333
Precision (minority class): 0.0
Recall (minority class): 0.0
F1 (minority class): 0.0


**This is the key lesson:** the dummy classifier that always predicts "not Platinum" scores ~87% accuracy — looking successful — but its precision, recall, and F1 for the minority class are all **0.0**, meaning it is completely unable to identify any Platinum member. This is why **precision, recall, F1-score, and the confusion matrix** (not accuracy alone) are the metrics that actually matter on imbalanced data.

In [7]:
confusion_matrix(y_test, dummy_predictions)

array([[265,   0],
       [ 35,   0]])

## 5. Undersampling

Undersampling reduces the number of majority-class examples to match the minority class, shrinking the dataset. It is fast and reduces training time, but discards potentially useful majority-class data, which can hurt the model's ability to learn the majority class well.

## 6. Oversampling

Oversampling increases the number of minority-class examples (by duplicating or synthesizing new ones) to match the majority class. It keeps all original data but increases dataset size and training time, and naive duplication can encourage overfitting to the specific minority examples that got duplicated.

## 7. Random Oversampling

Random oversampling duplicates existing minority-class rows at random until the class counts are balanced. It is simple and preserves all original information, but since it only copies existing rows, it adds no new information and can make the model overfit those exact duplicated points.

In [9]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
y_ros.value_counts()

is_platinum
0    617
1    617
Name: count, dtype: int64

## 8. Random Undersampling

Random undersampling randomly removes majority-class rows until the class counts are balanced. It's simple and fast, but risks discarding informative majority-class examples, which matters more when the dataset is already small (as with only ~700 training rows here).

In [10]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
y_rus.value_counts()

is_platinum
0    83
1    83
Name: count, dtype: int64

## 9. SMOTE (Synthetic Minority Over-sampling Technique)

SMOTE creates **new, synthetic** minority-class examples by interpolating between existing minority-class points and their nearest minority-class neighbors, rather than simply duplicating rows. This introduces more variety than random oversampling and tends to generalize better.

**When to use:** continuous/numeric features where interpolation between points makes sense, and when random oversampling's exact duplication is causing overfitting.

In [11]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
y_smote.value_counts()

is_platinum
0    617
1    617
Name: count, dtype: int64

## 10. Borderline-SMOTE

Borderline-SMOTE is a variant of SMOTE that focuses on generating synthetic samples specifically near the **decision boundary** between classes (the "borderline" minority points that are more likely to be misclassified), rather than interpolating uniformly across all minority points. This can produce more useful synthetic examples for the model to learn a sharper boundary.

**When to use:** when the classes overlap significantly and misclassifications tend to cluster near the boundary, since it targets the region that matters most for the decision surface.

In [12]:
from imblearn.over_sampling import BorderlineSMOTE
borderline_smote = BorderlineSMOTE(random_state=42, k_neighbors=5)
X_bsmote, y_bsmote = borderline_smote.fit_resample(X_train, y_train)
y_bsmote.value_counts()

is_platinum
0    617
1    617
Name: count, dtype: int64

## 11. Class Weights

Instead of changing the dataset itself, class weighting adjusts the model's **loss function** to penalize mistakes on the minority class more heavily than mistakes on the majority class. This keeps the original data untouched (no synthetic points, no discarded rows) while still pushing the model to pay attention to the minority class.

**When to use:** when you want to avoid altering the dataset (e.g. to preserve exact real-world distributions for evaluation), or when the model you're using supports a `class_weight` parameter directly (many scikit-learn classifiers do).

In [13]:
from sklearn.linear_model import LogisticRegression
weighted_model = LogisticRegression(class_weight='balanced', max_iter=1000)
weighted_model.fit(X_train, y_train)
weighted_predictions = weighted_model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, weighted_predictions))
print('Precision (minority class):', precision_score(y_test, weighted_predictions, zero_division=0))
print('Recall (minority class):', recall_score(y_test, weighted_predictions, zero_division=0))
print('F1 (minority class):', f1_score(y_test, weighted_predictions, zero_division=0))

Accuracy: 0.5366666666666666
Precision (minority class): 0.10606060606060606
Recall (minority class): 0.4
F1 (minority class): 0.16766467065868262


Compared to the dummy classifier's 0.0 precision/recall/F1, the class-weighted model trades a bit of raw accuracy for the ability to actually detect some Platinum members — this trade-off is usually worthwhile when the minority class is the one we care about identifying.

## 12. When to Use Each Technique

In [14]:
technique_comparison = pd.DataFrame({
    'technique': ['Random undersampling', 'Random oversampling', 'SMOTE', 'Borderline-SMOTE', 'Class weights'],
    'changes_dataset': ['yes (removes rows)', 'yes (duplicates rows)', 'yes (adds synthetic rows)', 'yes (adds synthetic rows near boundary)', 'no'],
    'risk': ['loses majority-class information', 'overfits duplicated points', 'may create unrealistic points in sparse regions', 'more targeted, but still synthetic', 'model must support weighting; no extra data diversity'],
    'best_when': [
        'majority class has abundant, redundant examples',
        'dataset is small and you cannot afford to lose majority rows',
        'features are continuous and interpolation is meaningful',
        'classes overlap heavily near the decision boundary',
        'you want to preserve the true data distribution untouched',
    ],
})
technique_comparison

,technique,changes_dataset,risk,best_when
0,Random undersampling,yes (removes rows),loses majority-class information,"majority class has abundant, redundant examples"
1,Random oversampling,yes (duplicates rows),overfits duplicated points,dataset is small and you cannot afford to lose...
2,SMOTE,yes (adds synthetic rows),may create unrealistic points in sparse regions,features are continuous and interpolation is m...
3,Borderline-SMOTE,yes (adds synthetic rows near boundary),"more targeted, but still synthetic",classes overlap heavily near the decision boun...
4,Class weights,no,model must support weighting; no extra data di...,you want to preserve the true data distributio...


## 13. Summary

- The Platinum-membership target in this dataset is imbalanced (~87% / ~13%), and the `purchase_amount_clean > 500` example shows an even more extreme imbalance (~99.4% / ~0.6%).
- A model that always predicts the majority class scores misleadingly high accuracy while being useless for the minority class — precision, recall, F1, and the confusion matrix are the metrics to trust instead.
- Undersampling, random oversampling, SMOTE, Borderline-SMOTE, and class weighting all address imbalance in different ways with different trade-offs; the right choice depends on dataset size, feature types, and whether preserving the original data distribution matters for evaluation.

In [14]:
df.to_csv('customer_transactions_imbalance_reviewed.csv', index=False)
df.shape

(1000, 16)